# OpenInfer Stage 1: Colab inference baseline

Run this notebook with a Colab GPU runtime if available. It loads one small instruct model, generates a fixed response, records basic performance details, and saves the measurements as JSON. The first model load downloads weights and may take several minutes.

## 1. Install dependencies

Run this cell once per fresh Colab runtime.

In [ ]:
%pip -q install -U transformers accelerate

## 2. Check the runtime

In Colab, choose **Runtime → Change runtime type → T4 GPU** before continuing, if a T4 is available. If this reports CPU, the notebook can still run, but that result is only a functional baseline.

In [ ]:
import platform
import time
from datetime import datetime, timezone
from pathlib import Path
import json

import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

use_cuda = torch.cuda.is_available()
device = 'cuda' if use_cuda else 'cpu'
dtype = torch.float16 if use_cuda else torch.float32
gpu_name = torch.cuda.get_device_name(0) if use_cuda else None
print(f'Device: {device}' + (f' ({gpu_name})' if gpu_name else ''))
print(f'PyTorch: {torch.__version__} | Transformers: {transformers.__version__}')

## 3. Load the model

The default is the roadmap's example model. Change `MODEL_ID` if needed, then rerun this cell and the cells below it.

In [ ]:
MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
PROMPT = 'Explain how a KV cache helps autoregressive text generation in three sentences.'
MAX_NEW_TOKENS = 96
WARMUP_RUNS = 1
MEASURED_RUNS = 3

load_started = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    low_cpu_mem_usage=True,
).to(device)
model.eval()
if use_cuda:
    torch.cuda.synchronize()
model_load_seconds = time.perf_counter() - load_started
print(f'Model loaded in {model_load_seconds:.2f}s')

## 4. Generate and measure

The measured duration covers a complete non-streaming generation. It is not time-to-first-token (TTFT); we can measure streaming metrics after the API is in place.

In [ ]:
messages = [{'role': 'user', 'content': PROMPT}]
rendered_prompt = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
inputs = tokenizer(rendered_prompt, return_tensors='pt').to(device)
prompt_tokens = int(inputs['input_ids'].shape[-1])

def generate_once():
    if use_cuda:
        torch.cuda.synchronize()
    started = time.perf_counter()
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
        )
    if use_cuda:
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - started
    generated_ids = output_ids[0, inputs['input_ids'].shape[-1]:]
    generated_tokens = int(generated_ids.numel())
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    return elapsed, generated_tokens, generated_text

for _ in range(WARMUP_RUNS):
    generate_once()

if use_cuda:
    torch.cuda.reset_peak_memory_stats()

run_records = []
for run_index in range(MEASURED_RUNS):
    elapsed, generated_tokens, generated_text = generate_once()
    run_records.append({
        'run': run_index + 1,
        'generation_seconds': elapsed,
        'generated_tokens': generated_tokens,
        'output_tokens_per_second': generated_tokens / elapsed if elapsed else None,
        'generated_text': generated_text,
    })
    print(f"Run {run_index + 1}: {elapsed:.2f}s, {generated_tokens} tokens, {generated_tokens / elapsed:.2f} tokens/s")

print('Sample output:\n' + run_records[-1]['generated_text'])

## 5. Save the baseline

Download the JSON result and add it to the project when you return to VS Code. Keep the hardware and settings with the result so future comparisons are meaningful.

In [ ]:
peak_gpu_memory_bytes = torch.cuda.max_memory_allocated() if use_cuda else None
result = {
    'recorded_at_utc': datetime.now(timezone.utc).isoformat(),
    'model_id': MODEL_ID,
    'prompt': PROMPT,
    'generation': {
        'max_new_tokens': MAX_NEW_TOKENS,
        'do_sample': False,
        'use_cache': True,
        'prompt_tokens': prompt_tokens,
        'warmup_runs': WARMUP_RUNS,
        'measured_runs': MEASURED_RUNS,
    },
    'environment': {
        'python': platform.python_version(),
        'platform': platform.platform(),
        'torch': str(torch.__version__),
        'transformers': transformers.__version__,
        'device': device,
        'gpu': gpu_name,
        'dtype': str(dtype).removeprefix('torch.'),
        'peak_gpu_memory_bytes': peak_gpu_memory_bytes,
    },
    'model_load_seconds': model_load_seconds,
    'runs': run_records,
}

output_path = Path('openinfer_baseline.json')
output_path.write_text(json.dumps(result, indent=2) + '\n', encoding='utf-8')
print(f'Saved {output_path}')

try:
    from google.colab import files
    files.download(str(output_path))
except ImportError:
    print('Not running in Colab; retrieve the JSON from the current working directory.')